# Mapa de México que muestra las zonas de mas alto riesgo a nivel estatal

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.express as px
import plotly.graph_objects as go

ruta_mapa = "data/mapa_mexico.json" 
ruta_vial = "data/red_vial.shp"

#GeoDataFrame
gdf = gpd.read_file(ruta_mapa)
red_vial = gpd.read_file(ruta_vial)



In [ ]:
#Ver las primeras lineas del GDF
print(gdf.head())
print(f'\nSistema de coordenadas')
print(gdf.crs)
print(gdf.columns)


In [ ]:



# Aseguramos CRS correcto
print("CRS:", gdf.crs)

# Ver cuántos municipios contiene
print("Número de municipios:", len(gdf))

# Visualización básica
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, color="#e0e0e0", edgecolor="black", linewidth=0.3)

ax.set_title("Mapa de la República Mexicana por municipio", fontsize=14, pad=15)
ax.set_axis_off()
plt.show()

In [ ]:
print(gdf['NOM_ENT'].unique())

In [ ]:
for estado in gdf['NOM_ENT'].unique():

    gdf_estado = gdf[gdf["NOM_ENT"] == estado]

    gdf_estado.plot(edgecolor="black", color="#c7e9b4", figsize=(6,6))
    plt.title(f"Municipios de {estado}")
    plt.axis("off")
    plt.show()

In [ ]:
delitos_df = pd.read_csv("data/Fuero_federal_2012-2015_sep2025.csv",  encoding="latin-1")

#RELLENAR LOS NaN CON CERO DE ENERO A DICIEMBRE
delitos_df.loc[:, "ENERO":"DICIEMBRE"] = (delitos_df.loc[:, "ENERO":"DICIEMBRE"].fillna(0)) 
delitos_df.head()


In [ ]:
#Se observa que en la columna INEGI aparecen NaN, esto por que corresponden al extranjero, vamos a excluirlos, debido a que no podremos mapearlos dentro del analisis
delitos_df[delitos_df['INEGI'].isna()]

#Excluimos con el operador diferente !=
delitos_df = delitos_df[delitos_df["ENTIDAD"] != "EXTRANJERO"]

In [ ]:
delitos_df.isna().sum()

In [ ]:
delitos_df['CONCEPTO'].unique()


In [ ]:
delitos_df['TIPO'].unique()

In [ ]:
delitos_df["TOTAL"] = delitos_df.loc[:, "ENERO":"DICIEMBRE"].sum(axis=1)
delitos_df

In [ ]:
#Veamos la columna entidad para el dataframe de delitos

delitos_df['ENTIDAD'].unique()



In [ ]:
gdf['NOM_ENT'].unique()

In [ ]:
#Ambos son diferentes, asi que los vamos a homologar

reemplazos = {
    "Aguascalientes": "AGUASCALIENTES",
    "Baja California": "BAJA CALIFORNIA",
    "Baja California Sur": "BAJA CALIFORNIA SUR",
    "Campeche": "CAMPECHE",
    "Coahuila de Zaragoza": "COAHUILA",
    "Colima": "COLIMA",
    "Chiapas": "CHIAPAS",
    "Chihuahua": "CHIHUAHUA",
    "Ciudad de México": "CIUDAD DE MEXICO",
    "Durango": "DURANGO",
    "Guanajuato": "GUANAJUATO",
    "Guerrero": "GUERRERO",
    "Hidalgo": "HIDALGO",
    "Jalisco": "JALISCO",
    "México": "MEXICO",
    "Michoacán de Ocampo": "MICHOACAN",
    "Morelos": "MORELOS",
    "Nayarit": "NAYARIT",
    "Nuevo León": "NUEVO LEON",
    "Oaxaca": "OAXACA",
    "Puebla": "PUEBLA",
    "Querétaro": "QUERETARO",
    "Quintana Roo": "QUINTANA ROO",
    "San Luis Potosí": "SAN LUIS POTOSI",
    "Sinaloa": "SINALOA",
    "Sonora": "SONORA",
    "Tabasco": "TABASCO",
    "Tamaulipas": "TAMAULIPAS",
    "Tlaxcala": "TLAXCALA",
    "Veracruz de Ignacio de la Llave": "VERACRUZ",
    "Yucatán": "YUCATAN",
    "Zacatecas": "ZACATECAS"
}



In [ ]:
#Reemplazamos los nombres con el diccionario, homologando estos datos para despues poder unirlos con un merge

gdf["NOM_ENT"] = gdf["NOM_ENT"].replace(reemplazos)
gdf['NOM_ENT'].unique()



In [ ]:
#Tomemos solo los delitos de tipo carretera que veamos en delitos_df
TIPOS_RELEVANTES = [
    "VIAS DE COMUNICACION Y CORRESPONDENCIA",
    "LEY DE VIAS GENERALES DE COMUNICACION (L.V.G.C.)",
    "PATRIMONIALES"
]

#Filtremos

delitos_carretera = delitos_df[delitos_df["TIPO"].isin(TIPOS_RELEVANTES)]
delitos_carretera



In [ ]:
#Hagamos una agrupación por año y entidad

deli_carr_group = delitos_carretera.groupby(["AÑO","INEGI", "ENTIDAD"], as_index=False)["TOTAL"].sum()
deli_carr_group




In [ ]:
gdf_map = gdf.merge(deli_carr_group, left_on="NOM_ENT", right_on="ENTIDAD", how="left")
gdf_map["TOTAL"] = gdf_map["TOTAL"].fillna(0)
gdf_map



In [ ]:


# Filtrar solo un año
año_objetivo = 2025
gdf_año = gdf_map[gdf_map["AÑO"] == año_objetivo].copy()

# Opcional: simplificar geometrías para acelerar render
gdf_año["geometry"] = gdf_año["geometry"].simplify(tolerance=0.01)

# Crear choropleth
fig = px.choropleth(
    gdf_año,
    geojson=gdf_año.geometry,
    locations=gdf_año.index,
    color="TOTAL",
    hover_name="NOM_ENT",
    color_continuous_scale="Blues"
)

# Ajustar límites del mapa

fig.update_traces(marker_line_width=0.09, marker_line_color="black")

# --- Ajustar mapa y título ---
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    title={
        'text': f"Mapa de Delitos en Carreteras — México {año_objetivo}",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    title_font=dict(size=22, family="Arial", color="black"),
    
    # 👇 Tamaño personalizado
    width=1800,   # ancho en píxeles
    height=900,   # alto en píxeles
)

# Mostrar
fig.show()


In [ ]:
#Vemos el CRS
print("CRS red vial:", red_vial.crs)
#Corregimos al anterior
red_vial = red_vial.to_crs(epsg=4326)

print("CRS red vial:", red_vial.crs)

In [ ]:
red_vial

In [ ]:
red_vial.columns

In [ ]:
red_vial['TIPO_VIAL'].unique()

In [ ]:
#print(red_vial['VELOCIDAD'].unique())
print('\n')
#red_vial[red_vial['VELOCIDAD']== 'N/A']


In [ ]:
red_carreteras = red_vial[red_vial['TIPO_VIAL']== 'Carretera']
red_carreteras

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# --- 1️⃣ Filtrar solo un año ---
año_objetivo = 2025
gdf_año = gdf_map[gdf_map["AÑO"] == año_objetivo].copy()
gdf_año["geometry"] = gdf_año["geometry"].simplify(tolerance=0.01)

# --- 2️⃣ Crear mapa base con choropleth ---
fig = px.choropleth(
    gdf_año,
    geojson=gdf_año.geometry,
    locations=gdf_año.index,
    color="TOTAL",
    hover_name="NOM_ENT",
    color_continuous_scale="Reds"
)

# --- 3️⃣ Extraer coordenadas de todas las carreteras ---
lon_all = []
lat_all = []

for geom in red_carreteras.geometry:
    x, y = geom.xy
    lon_all.extend(x)
    lat_all.extend(y)
    lon_all.append(None)
    lat_all.append(None)

# --- 4️⃣ Agregar capa de carreteras sobre el choropleth ---
fig.add_trace(go.Scattergeo(
    lon=lon_all,
    lat=lat_all,
    mode="lines",
    line=dict(color="green", width=1.2),
    opacity=0.8,
    hoverinfo="none",
    showlegend=False
))

# --- 5️⃣ Ajustar visualización ---
fig.update_traces(marker_line_width=0.09, marker_line_color="black")
fig.update_geos(fitbounds="locations", visible=False)

# --- 6️⃣ Layout general ---
fig.update_layout(
    title={
        'text': f"Mapa de Delitos y Red Carretera — México {año_objetivo}",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    title_font=dict(size=22, family="Arial", color="black"),
    width=1200,
    height=800
)

fig.show()


# Ver delitos por número de habitantes
# Ver delitos por número de carreteras

In [ ]:
red_carreteras.columns

In [ ]:
#red_carreteras['CALIREPR'].unique()

principales_carreteras = red_carreteras[
(red_carreteras['ADMINISTRA'] == 'Federal')]
#|(red_carreteras['ADMINISTRA'] == 'Estatal')
#]
#(red_carreteras['ESTATUS'] == 'Habilitado') |
#(red_carreteras['CONDICION'] == 'En operación')|
#(red_carreteras['CONDICION'] == 'En construcción - abierto')


principales_carreteras

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# --- 1️⃣ Filtrar solo un año ---
año_objetivo = 2025
gdf_año = gdf_map[gdf_map["AÑO"] == año_objetivo].copy()
gdf_año["geometry"] = gdf_año["geometry"].simplify(tolerance=0.01)

# --- 2️⃣ Crear mapa base con choropleth ---
fig = px.choropleth(
    gdf_año,
    geojson=gdf_año.geometry,
    locations=gdf_año.index,
    color="TOTAL",
    hover_name="NOM_ENT",
    color_continuous_scale="Reds"
)

# --- 3️⃣ Extraer coordenadas de todas las carreteras ---
lon_all = []
lat_all = []

for geom in principales_carreteras.geometry:
    x, y = geom.xy
    lon_all.extend(x)
    lat_all.extend(y)
    lon_all.append(None)
    lat_all.append(None)

# --- 4️⃣ Agregar capa de carreteras sobre el choropleth ---
fig.add_trace(go.Scattergeo(
    lon=lon_all,
    lat=lat_all,
    mode="lines",
    line=dict(color="green", width=1.2),
    opacity=0.8,
    hoverinfo="none",
    showlegend=False
))

# --- 5️⃣ Ajustar visualización ---
fig.update_traces(marker_line_width=0.09, marker_line_color="black")
fig.update_geos(fitbounds="locations", visible=False)

# --- 6️⃣ Layout general ---
fig.update_layout(
    title={
        'text': f"Mapa de Delitos y Red Carreteras Federales  — México {año_objetivo}",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    title_font=dict(size=22, family="Arial", color="black"),
    width=1200,
    height=800
)

fig.show()


In [ ]:
#Filtremos aun mas el GDF

gdf_año = gdf_año[["NOM_ENT", "TOTAL", "geometry"]]
gdf_año.columns

In [ ]:
#Filtremos aún más el df de principales carreteras

principales_carreteras = principales_carreteras[['NOMBRE', 'LONGITUD', 'geometry']]

principales_carreteras.columns


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# --- 1️⃣ Filtrar solo un año ---
año_objetivo = 2025
gdf_año = gdf_map[gdf_map["AÑO"] == año_objetivo].copy()
principales_carreteras.copy()
gdf_año["geometry"] = gdf_año["geometry"].simplify(tolerance=0.00001)
principales_carreteras['geometry'] = principales_carreteras['geometry'].simplify(tolerance=0.000001, preserve_topology=True)


# --- 2️⃣ Crear mapa base con choropleth ---
fig = px.choropleth(
    gdf_año,
    geojson=gdf_año.geometry,
    locations=gdf_año.index,
    color="TOTAL",
    hover_name="NOM_ENT",
    color_continuous_scale="Reds"
)

# --- 3️⃣ Extraer coordenadas de todas las carreteras ---
lon_all = []
lat_all = []

for geom in principales_carreteras.geometry:
    x, y = geom.xy
    lon_all.extend(x)
    lat_all.extend(y)
    lon_all.append(None)
    lat_all.append(None)

# --- 4️⃣ Agregar capa de carreteras sobre el choropleth ---
fig.add_trace(go.Scattergeo(
    lon=lon_all,
    lat=lat_all,
    mode="lines",
    line=dict(color="green", width=1.2),
    opacity=0.8,
    hoverinfo="none",
    showlegend=False
))

# --- 5️⃣ Ajustar visualización ---
fig.update_traces(marker_line_width=0.09, marker_line_color="black")
fig.update_geos(fitbounds="locations", visible=False)

# --- 6️⃣ Layout general ---
fig.update_layout(
    title={
        'text': f"Mapa de Delitos y Red Carreteras Federales  — México {año_objetivo}",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    title_font=dict(size=22, family="Arial", color="black"),
    width=1200,
    height=800
)

fig.write_html("ruta/mapa_mexico_2025.html")
fig.write_html("ruta/mapa.html", include_plotlyjs="cdn")
fig.show()
